# 07 — Condition Balance Audit
**Project:** Causal-History Dependence in Robot Policy Recovery  
**Phase:** 4 (Pilot Quality Audit & Statistical Power Planning)  
**Objective:** Audit the experimental balance and scenario-by-scenario outcome concordance across Counterfactual Conditions B0, B2, and B4.

In [1]:
import os, sys, json
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
metrics_file = os.path.join(repo_root, "results", "pilot_recovery", "rollout_metrics.json")
with open(metrics_file) as f:
    metrics = json.load(f)

rec_eval = metrics["recovery_evaluation"]

scenarios = []
for pol in ["reactive_mlp", "history_gru"]:
    for s in rec_eval[pol]["scenarios"]:
        scenarios.append({
            "policy": pol,
            "seed": s["seed"],
            "dev_time": s["dev_time"],
            "B0_success": s["conditions"]["B0"]["success"],
            "B2_success": s["conditions"]["B2"]["success"],
            "B4_success": s["conditions"]["B4"]["success"],
            "B4_error": s["conditions"]["B4"]["error"],
        })

df = pd.DataFrame(scenarios)
print("Scenario-by-Scenario Counterfactual Outcomes (N=8 Admitted Scenarios):")
print(df.to_string(index=False))


Scenario-by-Scenario Counterfactual Outcomes (N=8 Admitted Scenarios):
      policy  seed  dev_time  B0_success  B2_success  B4_success  B4_error
reactive_mlp    70        44       False       False       False       0.0
reactive_mlp   112        57       False       False       False       0.0
reactive_mlp   119        83        True        True        True       0.0
reactive_mlp   126        47       False       False       False       0.0
reactive_mlp   154        57       False       False       False       0.0
 history_gru    77        87        True        True        True       0.0
 history_gru   126        44       False       False       False       0.0
 history_gru   168        66       False       False       False       0.0


## 1. Outcome Concordance
We compute the pairwise concordance rate between conditions B0, B2, and B4.

In [2]:
b0_b2_match = (df["B0_success"] == df["B2_success"]).mean()
b0_b4_match = (df["B0_success"] == df["B4_success"]).mean()
b2_b4_match = (df["B2_success"] == df["B4_success"]).mean()

print(f"B0 vs B2 Concordance: {b0_b2_match * 100:.1f}%")
print(f"B0 vs B4 Concordance: {b0_b4_match * 100:.1f}%")
print(f"B2 vs B4 Concordance: {b2_b4_match * 100:.1f}%")


B0 vs B2 Concordance: 100.0%
B0 vs B4 Concordance: 100.0%
B2 vs B4 Concordance: 100.0%


## 2. Confound Invariance
All B4 injections achieved exact state restoration error = 0.0.

In [3]:
max_err = df["B4_error"].max()
print(f"Maximum State Matching Error under B4: {max_err:.10f}")
assert max_err == 0.0, "State matching error non-zero!"
print("Zero physical initial state confound verified across all scenarios.")


Maximum State Matching Error under B4: 0.0000000000
Zero physical initial state confound verified across all scenarios.
